
# Thermodynamics Laboratory — Interactive Version

This notebook is a **macroscopic thermodynamics simulator**.

You will not see particles. Instead, you control macroscopic variables and observe
how the thermodynamic state changes.

The first interactive apparatus is a gas in a cylinder with a movable piston.

You can choose:

- number of particles \(N\)
- initial temperature \(T_i\)
- initial volume \(V_i\)
- final volume \(V_f\)
- process type: **adiabatic** or **isothermal**

The notebook then calculates the process and displays

$
P(V),\qquad T(V),\qquad E(V),\qquad W.
$

The sign convention is

$
\delta W=-P_{\rm ext}\,dV,
$

so $W>0$ means work is done **on** the system.


In [1]:

# This cell checks that the interactive widget machinery is available.
# In a normal Jupyter installation, ipywidgets is usually already installed.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

print("ipywidgets version:", widgets.__version__)
print("Interactive controls will appear below after executing the next cell.")


ipywidgets version: 8.1.9
Interactive controls will appear below after executing the next cell.



## Interactive Experiment 1 — A piston

**Prediction first:** Before changing the sliders, predict what should happen to
the temperature when you compress an adiabatic gas.

Then use the controls below.

> **Important:** The sliders are live Jupyter widgets. They are not part of the
> static notebook preview. To use them, open this file in Jupyter Notebook or
> JupyterLab and execute the cells. If you are viewing the notebook in a web
> preview that does not execute widgets, use JupyterLab or Voilà.


In [2]:

# Thermodynamic model
k_B = 1.0
C_V = 1.5 * k_B
gamma = 1.0 + k_B / C_V

def process_piston(N, T_initial, V_initial, V_final, process="Adiabatic", n=300):
    if V_final <= 0 or V_initial <= 0:
        raise ValueError("Volumes must be positive.")

    V = np.linspace(V_initial, V_final, n)

    if process == "Adiabatic":
        T = T_initial * (V_initial / V)**(gamma - 1.0)
    elif process == "Isothermal":
        T = np.full_like(V, T_initial)
    else:
        raise ValueError("Unknown process type.")

    E = C_V * N * T
    P = N * k_B * T / V

    dV = np.diff(V)
    P_mid = 0.5 * (P[:-1] + P[1:])
    dW = -P_mid * dV
    W = np.concatenate([[0.0], np.cumsum(dW)])

    return pd.DataFrame({"V": V, "T": T, "P": P, "E": E, "W": W})

def run_piston(N=100, T_initial=1.0, V_initial=10.0, V_final=5.0,
               process="Adiabatic"):
    df = process_piston(N, T_initial, V_initial, V_final, process)

    E0 = df.E.iloc[0]
    Ef = df.E.iloc[-1]
    W = df.W.iloc[-1]
    Q = (Ef - E0) - W

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(df.V, df.P)
    axes[0].set_xlabel("$V$")
    axes[0].set_ylabel("$P$")
    axes[0].set_title("$P$-$V$ path")
    axes[0].invert_xaxis()

    axes[1].plot(df.V, df["T"])
    axes[1].set_xlabel("$V$")
    axes[1].set_ylabel("$T$")
    axes[1].set_title("Temperature")
    axes[1].invert_xaxis()

    axes[2].plot(df.V, df.E)
    axes[2].set_xlabel("$V$")
    axes[2].set_ylabel("$E$")
    axes[2].set_title("Internal energy")
    axes[2].invert_xaxis()

    plt.tight_layout()
    plt.show()

    summary = pd.DataFrame({
        "Quantity": ["Initial E", "Final E", "Delta E", "Q", "W", "Q + W",
                     "First-law residual"],
        "Value": [E0, Ef, Ef-E0, Q, W, Q+W, (Ef-E0)-(Q+W)]
    })

    display(summary)

# Controls
N = widgets.IntSlider(
    value=100, min=10, max=500, step=10,
    description="N", continuous_update=False,
    style={"description_width": "120px"}
)
T_initial = widgets.FloatSlider(
    value=1.0, min=0.2, max=3.0, step=0.1,
    description="Initial T", continuous_update=False,
    style={"description_width": "120px"}
)
V_initial = widgets.FloatSlider(
    value=10.0, min=2.0, max=20.0, step=0.5,
    description="Initial V", continuous_update=False,
    style={"description_width": "120px"}
)
V_final = widgets.FloatSlider(
    value=5.0, min=1.0, max=20.0, step=0.5,
    description="Final V", continuous_update=False,
    style={"description_width": "120px"}
)
process = widgets.ToggleButtons(
    options=["Adiabatic", "Isothermal"],
    value="Adiabatic",
    description="Process",
    style={"description_width": "120px"}
)

controls = widgets.VBox([N, T_initial, V_initial, V_final, process])

interactive_plot = widgets.interactive_output(
    run_piston,
    {
        "N": N,
        "T_initial": T_initial,
        "V_initial": V_initial,
        "V_final": V_final,
        "process": process
    }
)

display(controls)
display(interactive_plot)


Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 1500x400 with 3 Axes>', '…


### Questions

1. For the adiabatic process, what happens to $T$ when $V$ decreases?
2. For the isothermal process, what changes and what remains constant?
3. Compare the work $W$ for the two paths.
4. In each case, verify
    $
   \Delta E=Q+W.
   $
5. Why can the same initial and final volumes have different values of $Q$ and $W$?



# Interactive Experiment 2 — Energy redistribution and entropy maximization

Two systems are isolated from the outside world, but they can exchange energy with
one another. They cannot exchange particles or volume.

Thus
$
E_{\rm total}=E_1+E_2=\text{constant}.
$


### Predict before running

Start with an uneven distribution of energy.

1. Will $E_1$ and $E_2$ remain fixed?
2. If not, what will they approach?
3. Does the total energy change?
4. Is there a quantity that increases during the process?

Do not assume a temperature-equilibration condition in advance. Use the simulation
to discover the behavior.

Then examine all allowed energy distributions satisfying
$
E_2=E_{\rm total}-E_1.
$

Plot
$
S_{\rm total}(E_1)
=
S_1(E_1)+S_2(E_{\rm total}-E_1).
$

### Discovery question

> **Which allowed energy distribution corresponds to equilibrium?**

Your goal is to discover that the equilibrium distribution is the one for which
$
\boxed{S_{\rm total}\text{ is maximum}.}
$



In [17]:

def temperature_from_E(E, N):
    return E / (C_V * N)


def entropy_ideal(E, V, N):
    T = temperature_from_E(E, N)
    return C_V * N * np.log(T) + N * k_B * np.log(V)


def energy_redistribution(N1, N2, E1_initial, E2_initial,
                          V1=10.0, V2=10.0, n=250, rate=5.0):

    E_total = E1_initial + E2_initial

    # For the ideal-gas model, equilibrium requires equal energy per particle.
    E1_eq = E_total * N1 / (N1 + N2)

    t = np.linspace(0, 1, n)

    # Simple phenomenological relaxation model.
    # Microscopic degrees of freedom are intentionally hidden.
    E1 = E1_eq + (E1_initial - E1_eq) * np.exp(-rate * t)
    E2 = E_total - E1

    S1 = entropy_ideal(E1, V1, N1)
    S2 = entropy_ideal(E2, V2, N2)
    S_total = S1 + S2

    T1 = temperature_from_E(E1, N1)
    T2 = temperature_from_E(E2, N2)

    return pd.DataFrame({
        "t": t,
        "E1": E1,
        "E2": E2,
        "T1": T1,
        "T2": T2,
        "S_total": S_total
    })


def entropy_landscape(N1, N2, E1_initial, E2_initial,
                      V1=10.0, V2=10.0):

    E_total = E1_initial + E2_initial

    # All allowed energy distributions at fixed total energy.
    E1 = np.linspace(0.01 * E_total, 0.99 * E_total, 700)
    E2 = E_total - E1

    S_total = entropy_ideal(E1, V1, N1) + entropy_ideal(E2, V2, N2)

    imax = np.argmax(S_total)

    return E1, E2, S_total, E1[imax], E2[imax]


def run_energy_experiment(N1=100, N2=100,
                          E1_initial=0, E2_initial=360):

    df = energy_redistribution(
        N1, N2, E1_initial, E2_initial
    )

    # ---------------------------------------------------------
    # 1. Energy redistribution
    # ---------------------------------------------------------
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(df["t"], df["E1"], label="$E_1$")
    ax.plot(df["t"], df["E2"], label="$E_2$")
    ax.set_xlabel("Scaled time")
    ax.set_ylabel("Energy")
    ax.set_title("Redistribution of energy")
    ax.legend()
    plt.show()

    # ---------------------------------------------------------
    # 2. Total entropy during the process
    # ---------------------------------------------------------
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(df["t"], df["S_total"])
    ax.set_xlabel("Scaled time")
    ax.set_ylabel("$S_{\\rm total}$")
    ax.set_title("Total entropy")
    plt.show()

    # ---------------------------------------------------------
    # 3. Entropy landscape over all allowed distributions
    # ---------------------------------------------------------
    E1, E2, S_total, E1_max, E2_max = entropy_landscape(
        N1, N2, E1_initial, E2_initial
    )

    E1_final = df["E1"].iloc[-1]
    E2_final = df["E2"].iloc[-1]

    # fig, ax = plt.subplots(figsize=(7, 4))
    # ax.plot(E1, S_total, label="$S_{\\rm total}(E_1)$")
    # ax.axvline(E1_max, linestyle="--", label="Entropy maximum")
    # ax.axvline(E1_final, linestyle=":", label="Final state")
    # ax.set_xlabel("$E_1$")
    # ax.set_ylabel("$S_{\\rm total}$")
    # ax.set_title("Which energy distribution maximizes entropy?")
    # ax.legend()
    # plt.show()

    # ---------------------------------------------------------
    # 4. Summary
    # ---------------------------------------------------------
    E_total_initial = df["E1"].iloc[0] + df["E2"].iloc[0]
    E_total_final = df["E1"].iloc[-1] + df["E2"].iloc[-1]

    display(pd.DataFrame({
        "Quantity": [
            "Initial E1",
            "Initial E2",
            "Initial total E",
            "Final E1",
            "Final E2",
            "Final total E",
            "Initial S_total",
            "Final S_total",
            "Change in S_total"
        ],
        "Value": [
            df["E1"].iloc[0],
            df["E2"].iloc[0],
            E_total_initial,
            E1_final,
            E2_final,
            E_total_final,
            df["S_total"].iloc[0],
            df["S_total"].iloc[-1],
            df["S_total"].iloc[-1] - df["S_total"].iloc[0]
        ]
    }))


# -------------------------------------------------------------
# Interactive controls
# -------------------------------------------------------------
N1_energy = widgets.IntSlider(
    value=100, min=10, max=500, step=10,
    description="N1", continuous_update=False,
    style={"description_width": "120px"}
)

N2_energy = widgets.IntSlider(
    value=100, min=10, max=500, step=10,
    description="N2", continuous_update=False,
    style={"description_width": "120px"}
)

E1_energy = widgets.FloatSlider(
    value=100, min=0.2, max=360, step=10,
    description="Initial E1", continuous_update=False,
    style={"description_width": "120px"}
)

E2_energy = widgets.FloatSlider(
    value=260, min=0.2, max=360, step=10,
    description="Initial E2", continuous_update=False,
    style={"description_width": "120px"}
)

energy_controls = widgets.VBox([
    N1_energy,
    N2_energy,
    E1_energy,
    E2_energy
])

energy_output = widgets.interactive_output(
    run_energy_experiment,
    {
        "N1": N1_energy,
        "N2": N2_energy,
        "E1_initial": E1_energy,
        "E2_initial": E2_energy,
    }
)

display(energy_controls)
display(energy_output)


Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 700x400 with 1 Axes>', 'i…


# 11. A separate extremum principle — minimum energy

The previous experiment held the total energy fixed. Under those constraints, the
equilibrium state is found by **maximizing entropy**.

A different set of constraints leads to a different extremum statement.

At fixed
$
S,\;V,\;N,
$
the equilibrium state can equivalently be characterized by

$
\boxed{E\text{ is minimized}.}
$

### Important distinction

These are not two competing rules:

$
\boxed{
\begin{array}{ccl}
E,V,N\ \text{fixed} &\Longrightarrow& S\ \text{maximum},\\[4pt]
S,V,N\ \text{fixed} &\Longrightarrow& E\ \text{minimum}.
\end{array}}
$

The controlled variables determine which thermodynamic potential is the natural
quantity to extremize.

The next experiment is deliberately simple: use two identical subsystems and explore
how the minimum-energy state at fixed total entropy is associated with equal
temperatures.


In [15]:

def entropy_from_E_V(E, V, N):
    """Ideal-gas entropy, up to an irrelevant additive constant."""
    T = E / (C_V * N)
    return C_V * N * np.log(T) + N * k_B * np.log(V)


def volume_extremum_experiment(
    constraint="Constant total energy",
    delta_V=0.0,
    N=100,
    V_total=20.0,
    T_ref=1.0
):
    V_half = V_total / 2.0

    if abs(delta_V) >= V_half:
        raise ValueError("Choose |delta V| < V_total/2.")

    V1 = V_half + delta_V
    V2 = V_half - delta_V

    # Reference equilibrium state: equal volumes and equal temperatures.
    E_ref_each = C_V * N * T_ref
    E_ref_total = 2.0 * E_ref_each

    # ---------------------------------------------------------
    # Case 1: fixed total energy
    #
    # For this pedagogical experiment, the two identical systems
    # share the fixed energy equally. We then vary the volume split.
    # ---------------------------------------------------------
    if constraint == "Constant total energy":
        E_total = E_ref_total
        E1 = E_total / 2.0
        E2 = E_total / 2.0

        S_total = (
            entropy_from_E_V(E1, V1, N)
            + entropy_from_E_V(E2, V2, N)
        )

        # Scan the allowed volume distributions to find the maximum.
        delta_scan = np.linspace(-0.95 * V_half, 0.95 * V_half, 500)
        V1_scan = V_half + delta_scan
        V2_scan = V_half - delta_scan

        S_scan = (
            entropy_from_E_V(E1, V1_scan, N)
            + entropy_from_E_V(E2, V2_scan, N)
        )

        i_max = np.argmax(S_scan)
        delta_star = delta_scan[i_max]
        S_star = S_scan[i_max]

        quantity_value = S_total
        extremum_value = S_star
        extremum_label = "Maximum total entropy"

        y_label = "$S_{\\rm total}$"
        title = "Constant total energy: entropy versus volume redistribution"

    # ---------------------------------------------------------
    # Case 2: fixed total entropy
    #
    # At each volume split, impose the same total entropy as the
    # symmetric reference state. For identical systems at equilibrium,
    # T1 = T2 = T, so the total energy follows directly from the
    # constant-entropy condition.
    # ---------------------------------------------------------
    elif constraint == "Constant total entropy":
        S_target = (
            entropy_from_E_V(E_ref_each, V_half, N)
            + entropy_from_E_V(E_ref_each, V_half, N)
        )

        # With T1 = T2 = T:
        # S_total = 2 C_V N ln(T) + N k_B ln(V1 V2)
        # Solve for T at fixed S_total.
        T_common = np.exp(
            (
                S_target
                - N * k_B * np.log(V1 * V2)
            ) / (2.0 * C_V * N)
        )

        E1 = C_V * N * T_common
        E2 = C_V * N * T_common
        E_total = E1 + E2

        delta_scan = np.linspace(-0.95 * V_half, 0.95 * V_half, 500)
        V1_scan = V_half + delta_scan
        V2_scan = V_half - delta_scan

        T_scan = np.exp(
            (
                S_target
                - N * k_B * np.log(V1_scan * V2_scan)
            ) / (2.0 * C_V * N)
        )

        E_scan = 2.0 * C_V * N * T_scan

        i_min = np.argmin(E_scan)
        delta_star = delta_scan[i_min]
        extremum_value = E_scan[i_min]

        quantity_value = E_total
        extremum_label = "Minimum total energy"

        y_label = "$E_{\\rm total}$"
        title = "Constant total entropy: energy versus volume redistribution"

    else:
        raise ValueError("Unknown constraint.")

    # ---------------------------------------------------------
    # Plot local state and extremum landscape
    # ---------------------------------------------------------
    # fig, ax = plt.subplots(figsize=(7, 4))

    # if constraint == "Constant total energy":
    #     ax.plot(delta_scan, S_scan)
    #     ax.plot(
    #         delta_star, S_star,
    #         marker="o",
    #         markersize=8,
    #         label="Maximum entropy"
    #     )
    #     ax.axvline(0.0, linestyle="--", alpha=0.6,
    #                label="Equal volumes")

    # else:
    #     ax.plot(delta_scan, E_scan)
    #     ax.plot(
    #         delta_star, extremum_value,
    #         marker="o",
    #         markersize=8,
    #         label="Minimum energy"
    #     )
    #     ax.axvline(0.0, linestyle="--", alpha=0.6,
    #                label="Equal volumes")

    # ax.axvline(delta_V, linestyle=":", linewidth=2,
    #            label=r"Current $\delta V$")
    # ax.set_xlabel(r"$\delta V$")
    # ax.set_ylabel(y_label)
    # ax.set_title(title)
    # ax.legend()
    # plt.show()

    # ---------------------------------------------------------
    # Display the current state
    # ---------------------------------------------------------
    T1 = E1 / (C_V * N)
    T2 = E2 / (C_V * N)

    S1 = entropy_from_E_V(E1, V1, N)
    S2 = entropy_from_E_V(E2, V2, N)

    display(pd.DataFrame({
        "Quantity": [
            "$V_1$",
            "$V_2$",
            "$E_1$",
            "$E_2$",
            "$E_{\\rm total}$",
            "$T_1$",
            "$T_2$",
            "$S_{\\rm total}$",
            extremum_label
        ],
        "Value": [
            V1,
            V2,
            E1,
            E2,
            E_total,
            T1,
            T2,
            S1 + S2,
            extremum_value
        ]
    }))


constraint = widgets.ToggleButtons(
    options=["Constant total energy", "Constant total entropy"],
    value="Constant total energy",
    description="Constraint:",
    style={"description_width": "120px"}
)

delta_V = widgets.FloatSlider(
    value=0.0,
    min=-9.0,
    max=9.0,
    step=0.1,
    description=r"$\delta V$:",
    continuous_update=False,
    style={"description_width": "120px"}
)

interactive_volume = widgets.interactive_output(
    volume_extremum_experiment,
    {
        "constraint": constraint,
        "delta_V": delta_V
    }
)

display(widgets.VBox([constraint, delta_V]))
display(interactive_volume)


Output()